# Qwen3 attention-head taxonomy

This notebook scores three non-exclusive functional families: targeted retrieval, induction, and successor candidates. Targeted retrieval measures query-to-target-span enrichment. Induction uses the standard A B ... A to B attention pattern. Successor uses predecessor attention as a Q/K proxy and must be confirmed with OV/logit attribution or causal ablation before making a mechanistic successor-head claim.

In [5]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
import pandas as pd

REPO_URL = 'https://github.com/Twist-Shan/Realistic_CoT_NiaH_Count.git'
# In Colab, set this after cloning if you use a different destination.
EXPLICIT_REPO_ROOT = os.environ.get('REALISTIC_NIAH_REPO')

candidates = []
if EXPLICIT_REPO_ROOT:
    candidates.append(Path(EXPLICIT_REPO_ROOT).expanduser())
cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents])
candidates.extend([
    Path('/content/Realistic_CoT_NiaH_Count'),
    Path('/content/drive/MyDrive/Realistic_CoT_NiaH_Count'),
    Path('/content/drive/MyDrive/Colab Notebooks/Realistic_CoT_NiaH_Count'),
])

REPO_ROOT = next(
    (path.resolve() for path in candidates if (path / 'src' / 'dataset_generation').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the Realistic_CoT_NiaH_Count checkout. In Colab run:\n'
        f'  !git clone {REPO_URL} /content/Realistic_CoT_NiaH_Count\n'
        'Then rerun this cell. For a Drive checkout, set before this cell:\n'
        '  %env REALISTIC_NIAH_REPO=/content/drive/MyDrive/YOUR_PATH/Realistic_CoT_NiaH_Count'
    )

src_dir = REPO_ROOT / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print('Using repository:', REPO_ROOT)
print('Using source tree:', src_dir)

from dataset_generation.qk_hook_attention.analyze_qk_qwen3 import load_cache_tokens, load_tensor
from dataset_generation.qk_hook_attention.head_taxonomy import HeadTaxonomyConfig, scan_qk_cache

FileNotFoundError: Could not locate the Realistic_CoT_NiaH_Count checkout. In Colab run:
  !git clone https://github.com/Twist-Shan/Realistic_CoT_NiaH_Count.git /content/Realistic_CoT_NiaH_Count
Then rerun this cell. For a Drive checkout, set before this cell:
  %env REALISTIC_NIAH_REPO=/content/drive/MyDrive/YOUR_PATH/Realistic_CoT_NiaH_Count

## 1. Choose an existing Q/K cache

The cache must contain metadata.json, input_ids.pt, attention_mask.pt, position_ids.pt, layer Q/K tensors, and qk_norm tensors. The existing Qwen3 capture notebook or single-example pipeline can generate it.

In [ ]:
CACHE_DIR = REPO_ROOT / 'outputs' / 'qk_cache' / 'input_0'  # EDIT
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'head_taxonomy'
LAYERS = None  # None reads target_layers from cache metadata
HEADS = None   # None scans every query head
DEVICE = 'cuda'  # use 'cpu' for small caches

required = ['metadata.json', 'input_ids.pt', 'position_ids.pt']
missing = [name for name in required if not (CACHE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing cache files in {CACHE_DIR}: {missing}')

## 2. Inspect token positions and define evidence

Use half-open target spans [start, end]. Retrieval query positions should be answer/retrieval tokens after the target. Induction positions can be None to auto-detect every valid repeated-token pattern. For successor tests, explicit [query, predecessor-key] pairs are safest; a predecessor-token-ID to successor-token-ID map is also supported.

In [ ]:
ids = load_tensor(CACHE_DIR / 'input_ids.pt')
ids = (ids[0] if ids.ndim == 2 else ids).tolist()
tokens = load_cache_tokens(CACHE_DIR) or [''] * len(ids)
token_table = pd.DataFrame({'position': range(len(ids)), 'token_id': ids, 'token': tokens})
display(token_table.head(200))

In [ ]:
ANALYSIS_SPEC = {
    'target_spans': [],                  # example: [[120, 145], [410, 433]]
    'retrieval_query_positions': [],     # example: [980, 981, 982]
    'induction_query_positions': None,   # None = auto-detect all valid positions
    'successor_query_positions': None,   # None = all positions
    'successor_token_map': {},           # example: {monday_id: tuesday_id}
    'successor_pairs': [],               # example: [[query_pos, predecessor_pos]]
}

CONFIG = HeadTaxonomyConfig.from_dict(json.loads((REPO_ROOT / 'configs' / 'head_taxonomy.json').read_text()))
print('Edit ANALYSIS_SPEC before a real targeted-retrieval or successor run.')

## 3. Scan and save scores

Attention is reconstructed one layer/head at a time, so the implementation does not retain a full heads-by-sequence-by-sequence tensor. Outputs include head_scores.csv, head_labels.csv, head_evidence.json, and run_metadata.json.

In [ ]:
scores = scan_qk_cache(
    cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR, analysis_spec=ANALYSIS_SPEC,
    layers=LAYERS, heads=HEADS, config=CONFIG, device=DEVICE,
)
display(scores.sort_values(['primary_family', 'layer', 'head']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), constrained_layout=True)
for ax, family in zip(axes, ['targeted_retrieval', 'induction', 'successor']):
    table = scores.pivot(index='layer', columns='head', values=f'{family}_lift')
    image = ax.imshow(table, aspect='auto', interpolation='nearest', cmap='magma')
    ax.set(title=f'{family} lift', xlabel='head', ylabel='layer')
    fig.colorbar(image, ax=ax)
plt.show()

display(scores[scores.primary_family != 'unclassified'].sort_values(
    ['primary_family', 'layer', 'head']
))

## Interpretation checklist

Treat labels as screening results, not identities. Confirm stability across examples and prompt controls. Compare against the causal-uniform baseline and inspect per-query evidence. For successor candidates, add OV/logit attribution or head ablation before reporting a successor head.